# ML StudyBuDDY: A science specific learning tutor

In [ ]:
!pip install transformers datasets peft accelerate sentencepiece evaluate bitsandbytes gradio

### Importing the libraries and the dataset

In [1]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    pipeline
)

from peft import LoraConfig, get_peft_model
import evaluate
import time

from datasets import load_dataset

dataset = load_dataset("sciq")
print(dataset)

# Quick check
print("\nExample:")
print(dataset['train'][0])




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 230.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 121.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 127.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.99M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})

Example:
{'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'distractor3': 'viruses', 'distractor1': 'protozoa', 'distractor2': 'gymnosperms', 'correct_answer': 'mesophilic organisms', 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F),

### Reformat Dataset to an Instruction-Response format

In [2]:
def format_example(example):
    instruction = f"Answer the following science question:\n{example['question']}"
    response = f"{example['correct_answer']}.\nExplanation: {example['support']}"
    return {"instruction": instruction, "response": response}

formatted_dataset = dataset['train'].map(format_example, remove_columns=dataset['train'].column_names)
print(formatted_dataset[0])



Map:   0%|          | 0/11679 [00:00<?, ? examples/s]

{'instruction': 'Answer the following science question:\nWhat type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'response': 'mesophilic organisms.\nExplanation: Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}


### Train & Validation Split

In [3]:

split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds = split['test']

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

Train size: 10511
Validation size: 1168


### Tokenizer

In [4]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    full_texts = []
    labels = []

    for ins, res in zip(examples["instruction"], examples["response"]):

        full_prompt = f"### Instruction:\n{ins}\n\n### Response:\n{res}"
        instruction_part = f"### Instruction:\n{ins}\n\n### Response:\n"

        tokenized_full = tokenizer(
            full_prompt,
            truncation=True,
            padding="max_length",
            max_length=512
        )

        tokenized_instruction = tokenizer(
            instruction_part,
            truncation=True,
            padding=False,
            max_length=512
        )

        input_ids = tokenized_full["input_ids"]
        attention_mask = tokenized_full["attention_mask"]

        label_ids = input_ids.copy()

        instruction_length = len(tokenized_instruction["input_ids"])

        # Mask instruction portion
        label_ids[:instruction_length] = [-100] * instruction_length

        labels.append(label_ids)
        full_texts.append(input_ids)

    return {
        "input_ids": full_texts,
        "attention_mask": [tokenizer(
            f"### Instruction:\n{ins}\n\n### Response:\n{res}",
            truncation=True,
            padding="max_length",
            max_length=512
        )["attention_mask"] for ins, res in zip(examples["instruction"], examples["response"])],
        "labels": labels,
    }


train_tokenized = train_ds.map(tokenize_function, batched=True)
val_tokenized = val_ds.map(tokenize_function, batched=True)



tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/10511 [00:00<?, ? examples/s]

Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

### LoRA Fine-Tuning Setup

In [5]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", dtype="auto")

# LoRA Configuration
lora_config = LoraConfig(
    r=8,                # Rank
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### Training Arguments

In [7]:
training_args = TrainingArguments(
    output_dir="./science_tutor_lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    eval_strategy="steps",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    push_to_hub=False,
)



### Trainer

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer
)



Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


### Training & Saving the Model

In [9]:
start = time.time()

trainer.train()

end = time.time()

print("Training time:", end-start, "seconds")

model.save_pretrained("./science_tutor_lora")
tokenizer.save_pretrained("./science_tutor_lora")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,3.257900,0.394547
100,0.369400,0.374237
150,0.366600,0.369336
200,0.356800,0.366885
250,0.370500,0.365131
300,0.343200,0.363960
350,0.349900,0.363093
400,0.342700,0.362340
450,0.376300,0.362120
500,0.350500,0.361226


Training time: 7388.580475091934 seconds


('./science_tutor_lora/tokenizer_config.json',
 './science_tutor_lora/special_tokens_map.json',
 './science_tutor_lora/chat_template.jinja',
 './science_tutor_lora/tokenizer.model',
 './science_tutor_lora/added_tokens.json',
 './science_tutor_lora/tokenizer.json')

### Load Fine-Tuned Model

In [10]:
fine_model = AutoModelForCausalLM.from_pretrained(
    "./science_tutor_lora",
    device_map="auto",
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


### Create Pipelines

In [11]:
base_pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer
)

fine_pipe = pipeline(
    "text-generation",
    model=fine_model,
    tokenizer=tokenizer
)

Device set to use cuda:0
Device set to use cuda:0


### Base vs Fine-Tuned Comparison

In [12]:
question = "What is photosynthesis?"

prompt = f"""### Instruction:
Answer this science question:
{question}

### Response:
"""

print("BASE MODEL:\n")
print(base_pipe(prompt, max_new_tokens=100)[0]["generated_text"])

print("\nFINE-TUNED MODEL:\n")
print(fine_pipe(prompt, max_new_tokens=100)[0]["generated_text"])

BASE MODEL:

### Instruction:
Answer this science question:
What is photosynthesis?

### Response:
the process of obtaining energy from light.
Explanation: Photosynthesis is the process of obtaining energy from light.

FINE-TUNED MODEL:

### Instruction:
Answer this science question:
What is photosynthesis?

### Response:
the process of converting water and sunlight into food.
Explanation: Photosynthesis is the process of converting water and sunlight into food. The process requires energy from the sun and the exchange of electrons.


### Evaluation Metrics (Rouge, BLEU)

In [16]:
%uv pip install -q rouge_score nltk

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [17]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

predictions = []
references = []

for example in val_ds.select(range(50)):

    prompt = f"""### Instruction:
{example['instruction']}

### Response:
"""

    output = fine_pipe(prompt, max_new_tokens=100)[0]["generated_text"]

    prediction = output.split("### Response:")[-1].strip()

    predictions.append(prediction)
    references.append(example["response"])

rouge_score = rouge.compute(
    predictions=predictions,
    references=references
)

bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("ROUGE:", rouge_score)
print("BLEU:", bleu_score)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


ROUGE: {'rouge1': np.float64(0.2830509024614911), 'rouge2': np.float64(0.14350812339141272), 'rougeL': np.float64(0.23004544339800204), 'rougeLsum': np.float64(0.23345666563630374)}
BLEU: {'bleu': 0.05687034863839522, 'precisions': [0.43806921675774135, 0.23578751164958062, 0.15791984732824427, 0.10752688172043011], 'brevity_penalty': 0.27789553252071825, 'length_ratio': 0.43849840255591055, 'translation_length': 2196, 'reference_length': 5008}


## Evaluation Results

The model was evaluated using ROUGE and BLEU metrics on the validation dataset.

Results:

ROUGE-1: 0.283  
ROUGE-2: 0.144  
ROUGE-L: 0.230  
BLEU: 0.057  

These results indicate that the fine-tuned model is able to generate scientifically relevant answers with moderate lexical and structural similarity to the reference responses.

Given the limited training time and parameter-efficient fine-tuning approach, these results demonstrate successful domain adaptation.

### Inference Demo

In [20]:
def ask(question):

    prompt = f"""### Instruction:
Answer this science question:
{question}

### Response:
"""

    result = fine_pipe(
        prompt,
        max_new_tokens=200,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )

    return result[0]["generated_text"].split("### Response:")[-1].strip()

print(ask("What is gravity?"))

the force of attraction.
Explanation: Gravity is the force of attraction between objects in a gravitational field.


### Gradio Deployment

In [21]:
import gradio as gr

def chatbot(question):

    return ask(question)

gr.Interface(

    fn=chatbot,

    inputs="text",

    outputs="text",

    title="Science Tutor Assistant",

    description="Fine-tuned TinyLlama using LoRA on SciQ dataset"

).launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://03184de57477a36c5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
